# sg-sem: build the data and train the model

The Skill Graph semantic checker (see `SKILL_GRAPH_SEMANTIC_MODEL_SPEC.md` in the repo). This notebook runs on a **Colab GPU** through the VS Code Colab extension:

1. clones the repo and runs our own graph library (Node) to build seed graphs and text jobs,
2. runs an open model **on this GPU** (no API) to write outlines, checker answers, plain-word phrases and gates,
3. assembles the labelled examples, split by graph,
4. fine-tunes the model and reports dev and test metrics against the 0.9 precision bar.
5. scores the hand-written **gold set** (`gold/`), which the model never saw: it decides what ships.

**Run it with `QUICK_LOOK = True` first** (about 15 minutes): it writes a few answers per round so their quality can be checked. Then set it to `False` and run all again; finished answers are kept, so nothing is redone.

Save the notebook when it finishes: its outputs are what gets reviewed.

In [48]:
REPO = "https://github.com/Saad-web-spec/DevScratchPad.git"
BRANCH = "sg-sem"  # the branch with the sg-sem code
GENERATOR = "Qwen/Qwen2.5-7B-Instruct"   # writes the text; runs here on the GPU
BASE = "cross-encoder/ms-marco-MiniLM-L12-v2"  # the encoder sg-sem is fine-tuned from: the same MiniLM-L12-H384 network, already trained on text pairs (the raw microsoft/MiniLM-L12-H384-uncased never was)
EPOCHS = 10                                 # 4 through minilm-5: H2 loss was still falling at the end
QUICK_LOOK = False                         # True: a few answers per round, to check quality first (done; the full data is on Drive)
TRAIN = not QUICK_LOOK                     # training waits for the full data
RUN = "msmarco-4"  # minilm-1: H1 missed buried problems, H2 at chance. minilm-2: H1 flagged honest number ledgers, H2 starved (68 positives). minilm-3: H2 read the input list, not the task (dropped/swapped negatives added). minilm-4: H2 only woke on an appended sentence (woven positives added). minilm-5: 9 in 10 woven answers unusable (bracketed mention, drafted twin); the raw encoder never learnt to match A with B, so the base is now the pair-trained checkpoint of the same network. msmarco-1: every full-list row was a negative, so H2 read the length of the input list (replaced positives added). msmarco-2: shortcut closed but 391 H2 positives were too few to learn the comparison (H2-only outlines, every input woven, H2 feature dropout). msmarco-3: 1,170 positives, gold AUROC 0.74, but 1 of 28 cuts at the threshold: the model never saw the step that was missing (H2 pairs now name a candidate step; no new text-model calls). See MEMORY.md

## 1. Storage and setup

In [49]:
import os, shutil, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/sg-sem"
except Exception as error:  # the VS Code extension may not support Drive mounting
    print("Google Drive not mounted:", error)
    print("Data stays on this runtime only: if it disconnects, the run starts again.")
    ROOT = "/content/sg-sem"
DATA = f"{ROOT}/data-quick" if QUICK_LOOK else f"{ROOT}/data"
RUNS = f"{ROOT}/runs"
os.makedirs(DATA, exist_ok=True); os.makedirs(RUNS, exist_ok=True)
print("data:", DATA)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
data: /content/drive/MyDrive/sg-sem/data


In [50]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
CODE = "/content/DevScratchPad"
if not os.path.exists(CODE):
    !git clone --depth 1 -b {BRANCH} {REPO} {CODE}
else:
    !git -C {CODE} pull --ff-only
!git -C {CODE} log --oneline -1
node = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip() if shutil.which("node") else ""
if not node.startswith(("v20", "v22", "v24")):
    !curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1 && apt-get install -y nodejs > /dev/null 2>&1
!node --version
%cd {CODE}
!npm ci --ignore-scripts --no-audit --no-fund --loglevel=error
!pip install -q -r ml/sg-sem/requirements.txt
PIPE = "npx vite-node scripts/semantic/pipeline.ts"
LIMIT = "--limit 48" if QUICK_LOOK else ""

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 33 (delta 26), reused 28 (delta 23), pack-reused 0 (from 0)
Unpacking objects: 100% (33/33), 36.31 KiB | 1.10 MiB/s, done.
From https://github.com/Saad-web-spec/DevScratchPad
   a576179..b1de246  sg-sem     -> origin/sg-sem
Updating a576179..b1de246
Fast-forward
 MEMORY.md                                         |   9 +
 SKILL_GRAPH_SEMANTIC_MODEL_SPEC.md                |  17 +-
 ml/sg-sem/gold/examples.jsonl                     | 127 ++--
 ml/sg-sem/make_notebook.py                        |   2 +-
 ml/sg-sem/sg_sem.ipynb                            | 861 +++++++++++++---------
 ml/sg-sem/sgsem/__init__.py                       |   2 +-
 ml/sg-sem/sgsem/data.py                           |   5 +-
 ml/sg-sem/sgsem/evaluate.py                       |   4 +-
 ml/sg-sem/sgsem/leak.py        

## 2. Round 1: workflow outlines → seed graphs

In [51]:
!{PIPE} round1 --out {DATA}
!cd ml/sg-sem && python -m sgsem.generate --requests {DATA}/round1.requests.jsonl --outputs {DATA}/round1.outputs.jsonl --model {GENERATOR} {LIMIT}

⠙round1: 520 outline requests → /content/drive/MyDrive/sg-sem/data/round1.requests.jsonl
⠙520 already answered, 0 to go


## 3. Round 2: checker answers, plain-word inputs, gates

In [52]:
!{PIPE} round2 --out {DATA}
!cd ml/sg-sem && python -m sgsem.generate --requests {DATA}/round2.requests.jsonl --outputs {DATA}/round2.outputs.jsonl --model {GENERATOR} {LIMIT}

⠙round2: 437 seeds (423 drafted; rejected {"json":3,"draft":94,"errors":0,"duplicate":0}), 10705 requests {"h1":8190,"h4":455,"h2":2060}
why outlines were rejected:
  46 × steps[N].type must be one of orchestrator, worker, verifier, synthesizer, agent, code, approval.
  12 × steps[N].foreach must be <step> or <step>.<key>, e.g. plan.files: the step whose JSON list this step runs over.
  11 × Step … fans out over …, so … must be in its after.
  3 × steps[N].id must start with a letter and use only letters, digits, _ or - (got …).
  3 × Step … reads from …, which is not a step. Use the ids of other steps, or leave after empty to read the request.
  1 × Step … fans out over …, but … has no … among its keys (laptopN-summary, laptopN-summary, laptopN-summary).
  1 × Step … fans out over …, but … has no … among its keys (length, tone).
  1 × Step … fans out over …, but … has no … among its keys (story).
⠙10705 already answered, 0 to go


## 4. Assemble the labelled examples

In [53]:
!{PIPE} assemble --out {DATA}
print(open(f"{DATA}/stats.json").read())

⠙assemble: 16594 examples from 437 seeds; dropped {"h1-honest_pass-json":404,"h1-honest_pass-no-evidence":249,"h1-honest_pass-length":142,"h1-no_evidence_pass-length":450,"h1-no_evidence_pass-json":159,"h1-no_evidence_pass-has-evidence":8,"h1-resolved_problem_pass-no-evidence":577,"h1-resolved_problem_pass-json":262,"h1-resolved_problem_pass-no-fix":239,"h1-resolved_problem_pass-length":168,"h1-buried_problem_pass-json":293,"h1-buried_problem_pass-not-a-problem":291,"h1-buried_problem_pass-open-count":267,"h1-buried_problem_pass-length":2,"h4-unreadable":17,"h1-buried_number_pass-not-a-problem":257,"h1-buried_number_pass-open-count":238,"h1-buried_number_pass-json":278,"h1-buried_number_pass-length":13,"h1-flag-lost":3,"h2-checker-error":1,"h1-honest_number_pass-json":124,"h1-honest_number_pass-length":1016,"h1-honest_number_pass-no-evidence":83,"h1-honest_number_pass-over-limit":13,"h1-honest_number_pass-open-count":4,"h2-replaced-none":303,"h2-weave-length":149,"h2-weave-off-task":43

In [54]:
# For review: raw answers of the text model, two per kind and style (newest first), and the assembled samples.
import json, collections
requests = {json.loads(l)["id"]: json.loads(l) for l in open(f"{DATA}/round2.requests.jsonl")}
shown = collections.Counter()
for line in reversed(open(f"{DATA}/round2.outputs.jsonl").readlines()):
    row = json.loads(line); request = requests.get(row["id"])
    if request is None:  # an answer to an older prompt
        continue
    kind = (request["kind"], request["meta"].get("style", request["meta"].get("mode", "")))
    if shown[kind] < 2:
        shown[kind] += 1
        print(f"--- {' '.join(filter(None, kind))} {request['meta']}")
        print(row["text"][:1200])
        print()
print(open(f"{DATA}/samples.md").read()[:12000])

--- h2 weave {'ref': 'CHECK EXPERIENCE', 'mode': 'weave', 'seed': 'draft-h2-outline-3-0', 'node': 'generate_report', 'pred': 'check_experience'}
{"task": "Summarize the cover letter's suitability using [experience findings], skills, and writing style. Encourage if passing; offer constructive criticism otherwise.", "task_without": "Summarize the cover letter's suitability using skills and writing style. Encourage if passing; offer constructive criticism otherwise."}

--- h2 weave {'ref': 'VERIFY REPORTS', 'mode': 'weave', 'seed': 'draft-h2-outline-3-0', 'node': 'generate_report', 'pred': 'verify_reports'}
{"task": "Review each claim in the cover letter based on [verified reports], and summarize suitability with feedback on experience, skills, and style.", "task_without": "Review each claim in the cover letter, and summarize suitability with feedback on experience, skills, and style."}

--- h1 honest_number_pass {'style': 'honest_number_pass', 'seed': 'draft-outline-25-0', 'node': 'compo

## 5. Train (runs when `QUICK_LOOK = False`)

In [55]:
if TRAIN:
    !cd ml/sg-sem && python -m sgsem.train --data {DATA}/examples.jsonl --out {RUNS}/{RUN} --base {BASE} --epochs {EPOCHS}
else:
    print("QUICK_LOOK is on: training waits for the full data.")

train examples: {'h1': 2056, 'h2': 4680, 'h3': 733, 'h4': 2792} dev: {'h1': 362, 'h2': 1195, 'h3': 102, 'h4': 424}
I0000 00:00:1792755942.172278   32689 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792755942.245570   32689 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792755944.565015   32689 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00

In [56]:
import json
if TRAIN:
    report = json.load(open(f"{RUNS}/{RUN}/metrics.json"))
    print(json.dumps({k: report[k] for k in ["test", "rules_baseline_test", "meets_bar_on_test", "minutes"]}, indent=1))
    for section in open(f"{RUNS}/{RUN}/errors.md").read().split("\n## ")[1:]:  # the worst dev examples, per head
        print("## " + section[:2500])

{
 "test": {
  "h1": {
   "n": 232,
   "temperature": 2.0,
   "precision_at_05": 0.9805194805194806,
   "recall_at_05": 0.9869281045751634,
   "f1_at_05": 0.9837133550488599,
   "auroc": 0.9931331182261934,
   "threshold": 0.5,
   "precision_at_threshold": 0.9805194805194806,
   "recall_at_threshold": 0.9869281045751634,
   "score": 0.9837133550488599,
   "sentence_f1": 0.8207547169811321
  },
  "h2": {
   "n": 470,
   "temperature": 2.6,
   "pairs": 899,
   "pair_auroc": 0.944645954896704,
   "right_candidate": 0.9477124183006536,
   "precision_at_05": 0.7935483870967742,
   "recall_at_05": 0.803921568627451,
   "f1_at_05": 0.7987012987012987,
   "auroc": 0.9313416218222305,
   "threshold": 0.9153000116348267,
   "precision_at_threshold": 0.9868421052631579,
   "recall_at_threshold": 0.49019607843137253,
   "score": 0.7987012987012987,
   "span_f1": 0.8797633013319287
  },
  "h3": {
   "n": 73,
   "temperature": 3.6,
   "precision_at_05": 0.9032258064516129,
   "recall_at_05": 0.84848

## 6. Gold set (decides what ships)

Hand-written graphs, checker answers and gates in `ml/sg-sem/gold/`, on topics outside the training topics and never touched by the text model. Scored with the run's own thresholds, so the numbers are what the connector would do. A head ships only at precision >= 0.9 here.

In [57]:
import json
!npx vite-node scripts/semantic/gold.ts
!cd ml/sg-sem && python -m sgsem.evaluate --run {RUNS}/{RUN} --data gold/examples.jsonl --out {RUNS}/{RUN}/gold
gold = json.load(open(f"{RUNS}/{RUN}/gold/metrics.json"))
print(json.dumps({k: gold[k] for k in ["metrics", "rules_baseline", "meets_bar"]}, indent=1))
for section in open(f"{RUNS}/{RUN}/gold/errors.md").read().split("\n## ")[1:]:  # the worst gold examples, per head
    print("## " + section[:3500])

⠙gold: 7 graphs, 173 examples → ml/sg-sem/gold/examples.jsonl
{"h1":{"label0":20,"label1":22},"h2":{"label0":40,"label1":28},"h4":{"label0":28,"label1":14,"label2":21}}
⠙WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1792756183.805686   33824 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792756183.879473   33824 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792756186.343108   33824 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floatin

In [58]:
# Which cue does H2 read? Every gold cut positive is scored as is and with one thing changed; the column whose p(cut) moves is the cue.
!cd ml/sg-sem && python -m sgsem.probe --run {RUNS}/{RUN} --data gold/examples.jsonl

h2: 68 gold rows, 28 positives
I0000 00:00:1792756196.707696   34001 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792756196.777536   34001 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792756199.210996   34001 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792756199.211524   34001 cudart_stub.cc:31] Could not find cuda drivers on your

In [59]:
# Does one side alone give the H2 label away on the generated rows? Classifiers on text A only, text B only, the 16 facts only; test AUROC per negative kind.
!cd ml/sg-sem && python -m sgsem.leak --data {DATA}/examples.jsonl

h2: 9513 rows
label kind          rows  no {{ref}} in A  blocks in A  lines in B  A words
0     as written    2248             0.00         1.05        1.30       33
0     drafted       1025             0.00         1.65        1.65       31
0     dropped        887             0.00         0.92        1.31       25
0     kept          1181             0.12         0.56        1.58       31
0     other         2647             0.05         0.73        1.22       29
1     cut            941             0.05         0.86        1.29       30
1     replaced       584             0.00         1.89        1.89       31
rows whose A has no reference at all: 323, of them positive: 0.13

test AUROC of a train-split classifier that sees one side only (0.5 = the side carries nothing, near 1 = the label is in that side alone):
all positives             vs kept     vs drafted     vs dropped  vs as written         vs all
text A only                  0.60           0.55           0.64           0.89